# Phase 1: PubMedBERT classifier on PubMedQA (dev only)

Fine-tunes `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` on the 400 train
questions with 5 seeds and scores the 100 dev questions. Settings are pre-registered in
`docs/experiment-protocol.md` (section 3) and `configs/classifier.yaml`.

* Runtime: **GPU, T4**. Expected time: roughly 30-45 minutes for all 5 seeds.
* Results and checkpoints go to Google Drive, so a disconnect loses at most the current seed.
  Re-running the training cell skips finished seeds.
* The test split is locked: nothing here can load or score it.

## Step 1: settings (re-run this cell after any session restart)

In [ ]:
REPO_URL = "https://github.com/AyushYadav-rgb477/pubmedqa-rag-benchmark.git"
REF = "phase/1-data-scorer"  # branch or tag to run
REPO_DIR = "/content/pubmedqa-rag-benchmark"
DRIVE_ROOT = "/content/drive/MyDrive/biomedqa"
RESULTS_DIR = f"{DRIVE_ROOT}/results/classifier"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints/classifier"
HF_CACHE = "/content/hf_cache"

## Step 2: check the GPU and mount Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")

## Step 3: get the code (public repo, no token)

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {REF} && (git symbolic-ref -q HEAD > /dev/null && git pull -q --ff-only origin {REF} || true)
!git log --oneline -1

## Step 4: install the pinned stack

torchvision / torchaudio are removed because Colab's builds conflict with torch 2.6.0.

In [ ]:
!pip uninstall -y -q torchvision torchaudio
!pip install -q "torch==2.6.0" "transformers==4.51.3" "accelerate==1.6.0"
!pip install -q -e ".[classifier,dev]"

## Step 5: check versions

If an assertion fails with a wrong torch version: **Runtime > Restart session**, then run
Step 1, Step 3 and this cell again (no need to reinstall).

In [ ]:
import accelerate
import torch
import transformers

print(
    "torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| accelerate",
    accelerate.__version__,
    "| cuda",
    torch.cuda.is_available(),
)
assert torch.__version__.startswith("2.6.0"), "restart the session, then Step 1, 3, 5"
assert transformers.__version__ == "4.51.3"
assert accelerate.__version__ == "1.6.0"
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

## Step 6: run the test suite on this stack (includes a tiny CPU training smoke test)

In [ ]:
!python -m pytest -q

## Step 7: verify the splits (must print `splits verified`)

In [ ]:
!python scripts/make_splits.py --cache-dir {HF_CACHE}

## Step 8: train 5 seeds and score dev

In [ ]:
!python scripts/train_classifier.py --config configs/classifier.yaml --out {RESULTS_DIR} --checkpoint-dir {CKPT_DIR} --cache-dir {HF_CACHE}

## Step 9: summary

In [ ]:
import json

with open(f"{RESULTS_DIR}/summary.json") as f:
    summary = json.load(f)
print(json.dumps({k: summary[k] for k in ("n_seeds", "accuracy", "macro_f1")}, indent=2))
for seed, s in summary["per_seed"].items():
    print(seed, s)

## Step 10 (optional): check that a saved checkpoint reproduces its dev predictions

Writes to `/content` only, not to the results folder.

In [ ]:
!python scripts/predict_classifier.py --checkpoint {CKPT_DIR}/seed0 --seed 0 --split dev --out /content/reload_seed0_dev.jsonl --cache-dir {HF_CACHE}

## Step 11: download the results

In [ ]:
!cd {DRIVE_ROOT}/results && zip -qr /content/classifier_results.zip classifier
from google.colab import files

files.download("/content/classifier_results.zip")

## What to send back

Unzip `classifier_results.zip` into the local repo so that the files land in
`results/classifier/` (`summary.json`, `seed0/` ... `seed4/`). Checkpoints stay on Drive and are
never committed; they are needed again in Phase 4.